# Import Statements

In [1]:
import session_organizer
from sentence_transformers import SentenceTransformer
import pandas as pd

## Step 1: Load and Examine Data

In [ ]:
# First, examine the Excel file structure
file_path = "1.29.25 Abstracts.xlsx"
df_temp = pd.read_excel(file_path)

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

In [ ]:
# Define your column selections based on the output above
TITLE_COLUMN = 'Submission Name'  # Update based on your file
ABSTRACT_COLUMN = 'Abstract-Character max 4000-Abstracts will only be used to evaluate quality of talk and topic. They will not be published or able to be edited later.'  # Update based on your file  
ID_COLUMN = 'Submission ID - 7 digits'  # Update based on your file

# Load the data using the session_organizer function
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

## Step 2: Load Embedding Model

In [ ]:
# Available embedding models
EMBEDDING_MODELS = {
    "all-MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "all-mpnet-base-v2": "sentence-transformers/all-mpnet-base-v2", 
    "paraphrase-MiniLM-L6-v2": "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "cde-small-v1": "jxm/cde-small-v1"
}

# Select model (change as needed)
selected_model = "all-MiniLM-L6-v2"
model_name = EMBEDDING_MODELS[selected_model]

print(f"Loading embedding model: {model_name}")
embedding_model = SentenceTransformer(model_name, trust_remote_code=True)

if hasattr(embedding_model, 'model_card_data') and embedding_model.model_card_data:
    base_model = getattr(embedding_model.model_card_data, 'base_model', 'Unknown')
    print(f"Base model: {base_model}")
else:
    print(f"Model loaded: {model_name}")

# Process Steps

## Load Data

In [2]:
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations("1.29.25 Abstracts.xlsx", 
                                                                                         Title_name='Submission Name', 
                                                                                         Abstract_name='Abstract-Character max 4000-Abstracts will only be used to evaluate quality of talk and topic. They will not be published or able to be edited later.', 
                                                                                         Abstract_ID_name='Submission ID - 7 digits')

## Perform the Embedding

In [3]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', trust_remote_code=True)
print(embedding_model.model_card_data.base_model)
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)

sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

In [4]:
df_presentation_similarities = session_organizer.calculate_similarity_matrix(df_presentation_embeddings, df, embedding_model)

In [5]:
# # Load the embeddings model
# embedding_model = SentenceTransformer('jxm/cde-small-v1', trust_remote_code=True)
# print(embedding_model.model_card_data.base_model)
# df_presentation_similarities, df_presentation_embeddings = embed_documents(df, topic_column, embedding_model)

## Remove Duplicates and Near-Duplicates

In [6]:
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df, df_presentation_similarities, df_presentation_embeddings = session_organizer.remove_duplicates(df, df_presentation_similarities, df_presentation_embeddings, threshold=similarity_threshold)


Found 42 near-duplicate presentations to remove (keeping highest index).
Indices to remove: [41, 77, 78, 156, 221, 223, 269, 279, 325, 354, 410, 459, 477, 497, 541, 543, 561, 621, 713, 882, 905, 999, 1015, 1027, 1028, 1056, 1114, 1139, 1147, 1148, 1152, 1179, 1201, 1351, 1385, 1390, 1399, 1438, 1509, 1510, 1527, 1557]

Final number of oral presentations: 1559
Final shape of similarities matrix: (1559, 1559)
Final shape of embeddings matrix: (1559, 384)


## Create Sessions

In [7]:
df, df_sessions, labels, metadata = session_organizer.create_sessions(df, df_presentation_similarities, df_presentation_embeddings, max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name="Session Code")
print(f"Created {metadata['n_clusters']} sessions with a total of {metadata['n_assigned_items']} presentations assigned.")
print(f"Unassigned presentations: {metadata['n_unassigned_items']}")
print(metadata)

Created 100 sessions with a total of 1559 presentations assigned.
Unassigned presentations: 0
{'n_clusters': 100, 'n_assigned_items': 1559, 'n_unassigned_items': 0, 'cluster_sizes': [19, 13, 15, 14, 18, 14, 11, 16, 15, 27, 11, 11, 14, 26, 13, 16, 13, 19, 16, 12, 15, 17, 22, 15, 20, 20, 17, 13, 16, 23, 14, 15, 19, 11, 13, 13, 16, 15, 17, 13, 11, 19, 10, 25, 15, 11, 17, 19, 38, 16, 25, 18, 16, 18, 14, 14, 12, 21, 15, 12, 15, 22, 15, 11, 16, 13, 16, 12, 19, 16, 15, 16, 16, 17, 12, 8, 10, 10, 24, 20, 14, 15, 12, 13, 8, 13, 17, 17, 11, 8, 9, 14, 24, 11, 16, 15, 17, 13, 13, 13], 'total_presentations': 1559, 'clustering_efficiency': 1.0}


## Analyze Sessions

- session_coherence = "Are presentations within this session similar?" (internal session quality)
- session_distinctiveness = "Is this session's topic unique compared to others?" (relative session positioning)
- presentation_session_fit = "Does this presentation match the topic of others in the session?" (presentation fit)

Session Coherence measures cluster cohesion. It reflects how tighly grouped the topic of presentations within the session are.

Session Distinctiveness measures how unique each session's topic is. High values mean the session has a clear, focused theme that's different from other sessions. Low values suggest either the session mixes different topics or overlaps too much with other sessions.

Presentation-Session Fit is an individual presentations's average similarity to other presentation in its session. Generically, it can be referred to as "within_cluster_fit", "cluster_membership_strength", or "local_cohesion_score".

In [8]:
# Add to DataFrame
df_sessions['session_coherence'] = session_organizer.calculate_avg_similarity(df_sessions, df_presentation_similarities.values)
df_sessions['session_distinctiveness'] = session_organizer.calculate_silhouette_scores(df_sessions, df_presentation_embeddings.values, labels)
df['presentation_session_fit'] = session_organizer.calculate_document_similarities(df_presentation_similarities.values, labels)

## Create Session Titles & Keywords

In [9]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

Ollama status: 200
Available models: ['llama3.2:latest']


In [10]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Using model: llama3.2:latest
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 46.12 seconds
Average time per session: 15.37 seconds
 cluster_id                                                                                   presentation_indices  cluster_size  session_coherence  session_distinctiveness                                                     Ollama Title 1                                                             Ollama Title 2                                                                         Ollama Title 3                                                                                                                                         Ollama Keywords
          0 [133, 146, 210, 243, 439, 452, 596, 667, 778, 877, 941, 980, 1110, 1116, 1121, 1129, 1210, 1258, 1288]            19           0.516

In [11]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

LLaMA model loaded successfully
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 571.0409 seconds
Average time per session: 190.3470 seconds
 cluster_id                                                                                   presentation_indices  cluster_size  session_coherence  session_distinctiveness                                                                           Llama Title 1                                                            Llama Title 2                                                                                  Llama Title 3                                                                                                                                                                                                                                                                     

In [12]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

Model variable 'model' not found, or already unloaded.


In [13]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 8.5864 seconds
Average time per session: 2.8621 seconds
 cluster_id                                                                                   presentation_indices  cluster_size  session_coherence  session_distinctiveness                                                              Gemini Title 1                                                Gemini Title 2                                                                    Gemini Title 3                                                                            Gemini Keywords
          0 [133, 146, 210, 243, 439, 452, 596, 667, 778, 877, 941, 980, 1110, 1116, 1121, 1129, 1210, 1258, 1288]            19           0.516857                 0.031459                   AI-Powered Precision Systems for Crop and Weed Man

## Match Committees to Related Sessions

In [14]:
# Read the file and create DataFrame
file_path = 'ASABE Committees.txt'  # Update this path as needed
df_committees = session_organizer.parse_committee_file_simple(file_path)
committee_topics = df_committees['Name_Description'].tolist()
df_committee_embeddings = session_organizer.embed_documents(df_committees, 'Name_Description', embedding_model)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [15]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings, 
    df_committees, 
    df_committee_embeddings.values, 
    top_n=3,
)

In [16]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")
sample_columns = ['cluster_id', 'Top Committee Match', 'Top Committee Similarity', 
                  '2nd Committee Match', '2nd Committee Similarity', 
                  '3rd Committee Match', '3rd Committee Similarity']
print(df_sessions[sample_columns].head(10).to_string(index=False))


Sample of top committee matches:
 cluster_id                             Top Committee Match Top Committee Similarity                                    2nd Committee Match 2nd Committee Similarity                        3rd Committee Match 3rd Committee Similarity
          0               MS-45 Soil-Plant-Machine Dynamics                 0.430897                         NRES-244 Irrigation Management                 0.424146   PRS-702 Crop & Feed Processing & Storage                 0.422592
          1                  NRES-244 Irrigation Management                 0.484914                          MS-60 Unmanned Aerial Systems                  0.44994   PRS-702 Crop & Feed Processing & Storage                 0.437702
          2        PRS-702 Crop & Feed Processing & Storage                 0.307294     MS-23/19/3 Electronics for Identification (Animal)                 0.296466  MS-23/7 Harvest and US TAG ISO/TC 23/SC 7                 0.289872
          3 NRES-25 Streams, Reser

In [17]:
# # save the dataframes to pickle files
# df_sessions.to_pickle('df_sessions.pkl')
# df.to_pickle('df_presentations.pkl')

In [18]:
# df_sessions_sample.to_pickle('df_sessions_sample.pkl')

## Save Session Data for Later Use

After creating sessions, save the data so you can load it later for title generation without re-running the embedding process.

In [ ]:
# Save session data to pickle file for later use
import pickle

# Prepare session data dictionary
session_data = {
    'df': df,
    'df_sessions': df_sessions,
    'df_presentation_embeddings': df_presentation_embeddings,
    'df_presentation_similarities': df_presentation_similarities,
    'labels': labels,
    'metadata': metadata,
    'topic_column': topic_column,
    'original_filepath': file_path
}

# Save to pickle file
session_data_file = "session_data.pkl"
with open(session_data_file, 'wb') as f:
    pickle.dump(session_data, f)

print(f"✓ Session data saved to: {session_data_file}")
print(f"✓ Sessions: {len(df_sessions)}")
print(f"✓ Presentations: {len(df)}")

## Load Session Data (Alternative Start Point)

If you have saved session data, you can load it and continue from title generation.

In [ ]:
# Load previously saved session data
import pickle

session_data_file = "session_data.pkl"

try:
    with open(session_data_file, 'rb') as f:
        session_data = pickle.load(f)
    
    # Extract variables from loaded data
    df = session_data['df']
    df_sessions = session_data['df_sessions']
    df_presentation_embeddings = session_data['df_presentation_embeddings']
    df_presentation_similarities = session_data['df_presentation_similarities']
    labels = session_data['labels']
    metadata = session_data['metadata']
    topic_column = session_data['topic_column']
    
    print(f"✓ Session data loaded successfully!")
    print(f"✓ Sessions: {len(df_sessions)}")  
    print(f"✓ Presentations: {len(df)}")
    print(f"✓ Topic column: {topic_column}")
    print("\nYou can now proceed directly to title generation.")
    
except FileNotFoundError:
    print("✗ Session data file not found. Please run the full process first.")
except Exception as e:
    print(f"✗ Error loading session data: {e}")

## Complete Session Creation Workflow

This workflow is now separated into two main phases:
1. **Session Creation**: Embedding, clustering, and analysis (no LLM required)
2. **Title Generation**: Requires LLM configuration (Gemini API or Ollama)

In [ ]:
# Phase 1: Complete session creation workflow (no LLM needed)
# This includes all steps up to session analysis and saving intermediate results

print("=" * 50)
print("PHASE 1: SESSION CREATION")
print("=" * 50)

# Steps 1-6: Load data through session analysis
# (Use existing cells 5-18 for this phase)

# After completing session analysis, save intermediate results
print("\nPhase 1 complete - sessions created and analyzed")
print("Ready for Phase 2: Title Generation")

## Phase 2: Title Generation (Requires LLM Configuration)

This phase requires either:
- Gemini API key for online generation
- Ollama server running for local generation

In [ ]:
# Phase 2: Title generation - requires LLM setup
print("=" * 50)
print("PHASE 2: TITLE GENERATION")
print("=" * 50)

# Choose your LLM method
USE_GEMINI = True  # Set to False to use Ollama instead

if USE_GEMINI:
    # Requires GEMINI_API_KEY in .env file
    model_name = "gemini-2.0-flash"
    print("Using Gemini API for title generation...")
else:
    # Requires Ollama server running
    model_name = "ollama:llama3.2:latest"
    print("Using Ollama for title generation...")
    
    # Test Ollama connection
    import requests
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            print("✓ Ollama server is accessible")
        else:
            print("✗ Ollama server not responding properly")
    except Exception as e:
        print(f"✗ Cannot connect to Ollama: {e}")
        print("Please run 'ollama serve' first")